In [ ]:
import os
import glob
import random
import warnings
import multiprocessing
import pandas as pd
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as T
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import WandbLogger
from torchmetrics.classification import MulticlassF1Score
from transformers import ASTModel, get_cosine_schedule_with_warmup
from pathlib import Path
from tqdm import tqdm

In [ ]:
!uv pip install --upgrade wandb

In [ ]:
# ==========================================
# 0. CONFIGURATION & SETUP
# ==========================================
warnings.filterwarnings('ignore')
pl.seed_everything(14, workers=True)

# Toggle pipeline stages here
RUN_PSEUDO_LABELING = True
RUN_TRAINING = True
RUN_INFERENCE = True

CONFIG = {
    "project_name": "23f2001635-t12026",
    "experiment_name": "ast_mashup_v2_pseudo_tta",
    "data_root": "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup",
    
    # Existing synthetic data from your V1 run
    "synthetic_data_dir": "/kaggle/input/datasets/harshitsahu01/dataset-for-ast-v1/processed_data_ast",
    
    # Where to save the new pseudo labels temporarily
    "pseudo_save_dir": "/kaggle/working/pseudo_labels",
    
    # Your best V1 checkpoint to use for labeling and backbone initialization
    "best_checkpoint_v1": "/kaggle/input/models/harshitsahu01/ast-genre-predictor/pytorch/default/1/checkpoints/best-ast-model.ckpt",
    
    "head_lr": 1e-3,         # Higher LR for the new FFN head
    "backbone_lr": 1e-5,     # Tiny LR for the unthawed AST backbone
    "weight_decay": 0.05,
    "warmup_epochs": 2,      # Epochs to keep AST backbone frozen
    "batch_size": 16,
    "accumulate_grad": 4,
    "num_epochs": 40,
    "patience": 5,
    "num_classes": 10,
    "sample_rate": 16000,
    "duration": 30,
    "ast_frames": 1024,
    "pseudo_threshold": 0.95 # Only accept predictions with > 95% confidence
}

GENRES = ["blues", "classical", "country", "disco", "hiphop", 
          "jazz", "metal", "pop", "reggae", "rock"]
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}

AST_MEAN = -4.2677393
AST_STD = 4.5689974

if RUN_TRAINING:
    import wandb
    from kaggle_secrets import UserSecretsClient
    try:
        user_secrets = UserSecretsClient()
        wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))
    except Exception as e:
        print(f"WandB login skipped/failed: {e}")

In [ ]:
# ==========================================
# 1. OLD MODEL DEFINITION (Needed to load V1 Checkpoint)
# ==========================================
class MessyMashupAST_v1(pl.LightningModule):
    def __init__(self, config):
        super().__init__()
        self.save_hyperparameters(config)
        self.ast = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
        self.classifier = nn.Sequential(nn.LayerNorm(768), nn.Dropout(0.5), nn.Linear(768, config["num_classes"]))
    def forward(self, x):
        outputs = self.ast(x)
        return self.classifier(outputs.last_hidden_state[:, 0, :])

# ==========================================
# 2. PSEUDO-LABELING GENERATOR
# ==========================================
def generate_pseudo_labels(config):
    print("--- GENERATING PSEUDO LABELS FROM TEST SET ---")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print(f"Loading V1 Model from {config['best_checkpoint_v1']}")
    model_v1 = MessyMashupAST_v1.load_from_checkpoint(config['best_checkpoint_v1']).to(device)
    model_v1.eval()
    
    test_df = pd.read_csv(f"{config['data_root']}/test.csv")
    success_count = 0
    
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        file_id = row['id']
        filename = row.get('filename', f"mashups/{file_id}.wav") 
        audio_path = f"{config['data_root']}/{filename}"
        
        try:
            y, _ = librosa.load(audio_path, sr=config["sample_rate"])
            target_samples = config["duration"] * config["sample_rate"]
            if len(y) < target_samples: y = np.pad(y, (0, target_samples - len(y)))
            else: y = y[:target_samples]

            mel_spec = librosa.feature.melspectrogram(y=y, sr=config["sample_rate"], n_mels=128, n_fft=400, hop_length=160)
            mel_spec_db = (librosa.power_to_db(mel_spec, ref=np.max) - AST_MEAN) / (AST_STD * 2)
            tensor = torch.tensor(mel_spec_db, dtype=torch.float32)
            
            # Predict on center crop
            frames = tensor.shape[1]
            if frames > config["ast_frames"]:
                start = (frames - config["ast_frames"]) // 2
                crop = tensor[:, start : start + config["ast_frames"]]
            else:
                crop = F.pad(tensor, (0, config["ast_frames"] - frames))
                
            input_tensor = crop.transpose(0, 1).unsqueeze(0).to(device)
            
            with torch.no_grad():
                logits = model_v1(input_tensor)
                probs = F.softmax(logits, dim=1)
                max_prob, pred_idx = torch.max(probs, dim=1)
                
            # If highly confident, save as new training data
            if max_prob.item() >= config["pseudo_threshold"]:
                genre = IDX_TO_GENRE[pred_idx.item()]
                
                # 90-10 Split Logic
                prefix = "train" if random.random() < 0.9 else "val"
                
                save_dir = Path(config["pseudo_save_dir"]) / prefix / genre
                save_dir.mkdir(parents=True, exist_ok=True)
                torch.save(tensor, save_dir / f"pseudo_{file_id}.pt")
                success_count += 1
                
        except Exception as e:
            continue
            
    print(f"✅ Generated {success_count} high-confidence pseudo-labels.")

In [ ]:
# ==========================================
# 3. DUAL-DIRECTORY DATASET
# ==========================================
class ASTMashupDatasetV2(Dataset):
    def __init__(self, data_dirs, config, is_train=True):
        self.files = []
        for d in data_dirs:
            self.files.extend(glob.glob(f"{d}/*/*.pt"))
            
        self.is_train = is_train
        self.target_frames = config["ast_frames"]
        self.time_masking = T.TimeMasking(time_mask_param=100)
        self.freq_masking = T.FrequencyMasking(freq_mask_param=24)

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        label = GENRE_TO_IDX[Path(file_path).parent.name]
        mel_spec = torch.load(file_path)
        
        total_frames = mel_spec.shape[1]
        if total_frames > self.target_frames:
            if self.is_train:
                start = random.randint(0, total_frames - self.target_frames)
            else:
                start = (total_frames - self.target_frames) // 2
            mel_spec = mel_spec[:, start : start + self.target_frames]
        else:
            mel_spec = F.pad(mel_spec, (0, self.target_frames - total_frames))

        mel_spec = mel_spec.transpose(0, 1)

        if self.is_train:
            mel_spec = mel_spec.transpose(0, 1).unsqueeze(0)
            mel_spec = self.time_masking(mel_spec)
            mel_spec = self.freq_masking(mel_spec)
            mel_spec = mel_spec.squeeze(0).transpose(0, 1)

        return mel_spec, torch.tensor(label, dtype=torch.long)

In [ ]:
# ==========================================
# 4. V2 ARCHITECTURE (Freeze-Thaw + FFN Head)
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        return (self.alpha * (1 - pt)**self.gamma * ce_loss).mean()

class MessyMashupAST_v2(pl.LightningModule):
    def __init__(self, config, pretrained_v1_model=None):
        super().__init__()
        self.save_hyperparameters(config)
        self.config = config
        
        # 1. Load the finely-tuned AST backbone from V1
        if pretrained_v1_model:
            self.ast = pretrained_v1_model.ast
        else:
            self.ast = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
            
        # 2. Attach the new robust Feed-Forward Network
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, config["num_classes"])
        )

        self.criterion = FocalLoss(gamma=2.0)
        self.train_f1 = MulticlassF1Score(num_classes=config["num_classes"], average='macro')
        self.val_f1 = MulticlassF1Score(num_classes=config["num_classes"], average='macro')

    def forward(self, x):
        outputs = self.ast(x)
        return self.classifier(outputs.last_hidden_state[:, 0, :])

    def on_train_epoch_start(self):
        # Freeze-Thaw Logic
        if self.current_epoch < self.config["warmup_epochs"]:
            for param in self.ast.parameters(): param.requires_grad = False
            if self.current_epoch == 0: print("\n❄️ AST Backbone FROZEN for Warmup.")
        else:
            for param in self.ast.parameters(): param.requires_grad = True
            if self.current_epoch == self.config["warmup_epochs"]: print("\n🔥 AST Backbone THAWED for Fine-tuning.")

    def training_step(self, batch, batch_idx):
        x, y = batch
        # Mixup
        if np.random.rand() < 0.5:
            alpha = 0.4
            lam = np.random.beta(alpha, alpha)
            rand_index = torch.randperm(x.size()[0]).cuda()
            target_a, target_b = y, y[rand_index]
            x = lam * x + (1 - lam) * x[rand_index]
            logits = self(x)
            loss = lam * self.criterion(logits, target_a) + (1 - lam) * self.criterion(logits, target_b)
        else:
            logits = self(x)
            loss = self.criterion(logits, y)

        preds = torch.argmax(logits, dim=1)
        self.train_f1(preds, y)
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_macro_f1', self.train_f1, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        preds = torch.argmax(logits, dim=1)
        self.val_f1(preds, y)
        self.log('val_loss', self.criterion(logits, y), prog_bar=True)
        self.log('val_macro_f1', self.val_f1, prog_bar=True)

    def configure_optimizers(self):
        # Differential Learning Rates
        optimizer = torch.optim.AdamW([
            {'params': self.ast.parameters(), 'lr': self.config["backbone_lr"]},
            {'params': self.classifier.parameters(), 'lr': self.config["head_lr"]}
        ], weight_decay=self.config["weight_decay"])
        
        total_steps = self.trainer.estimated_stepping_batches
        scheduler = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps
        )
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "interval": "step"}}

In [ ]:
# ==========================================
# 5. INFERENCE (TEST-TIME AUGMENTATION)
# ==========================================
def run_inference_tta(model, config):
    print("--- RUNNING INFERENCE WITH TTA ---")
    model.eval()
    model.cuda()
    
    test_df = pd.read_csv(f"{config['data_root']}/test.csv")
    predictions = []
    
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        file_id = row['id']
        filename = row.get('filename', f"mashups/{file_id}.wav") 
        audio_path = f"{config['data_root']}/{filename}"
        
        try:
            y, _ = librosa.load(audio_path, sr=config["sample_rate"])
            mel_spec = librosa.feature.melspectrogram(y=y, sr=config["sample_rate"], n_mels=128, n_fft=400, hop_length=160)
            mel_spec_db = (librosa.power_to_db(mel_spec, ref=np.max) - AST_MEAN) / (AST_STD * 2)
            tensor = torch.tensor(mel_spec_db, dtype=torch.float32)
            
            total_frames = tensor.shape[1]
            frames = config["ast_frames"]
            
            if total_frames > frames:
                # TTA: Extract Start, Center, and End crops
                crop1 = tensor[:, 0 : frames]
                crop2 = tensor[:, (total_frames - frames) // 2 : ((total_frames - frames) // 2) + frames]
                crop3 = tensor[:, total_frames - frames : total_frames]
                
                batch = torch.stack([crop1.transpose(0, 1), crop2.transpose(0, 1), crop3.transpose(0, 1)]).cuda()
            else:
                crop = F.pad(tensor, (0, frames - total_frames))
                batch = crop.transpose(0, 1).unsqueeze(0).cuda()

            with torch.no_grad():
                logits = model(batch)
                # Average the probabilities across the TTA crops
                avg_probs = torch.mean(F.softmax(logits, dim=1), dim=0)
                pred_idx = torch.argmax(avg_probs).item()
                predictions.append(IDX_TO_GENRE[pred_idx])
                
        except Exception as e:
            print(f"Failed on {file_id}: {e}")
            predictions.append("pop") 

    test_df['genre'] = predictions
    test_df[['id', 'genre']].to_csv('submission_v2_tta.csv', index=False)
    print("Saved predictions to submission_v2_tta.csv")

In [ ]:

# ==========================================
# 6. EXECUTION PIPELINE
# ==========================================
if __name__ == "__main__":
    if RUN_PSEUDO_LABELING:
        generate_pseudo_labels(CONFIG)

    if RUN_TRAINING:
        print("--- STARTING V2 TRAINING ---")
        
        # Combine directories!
        train_dirs = [f"{CONFIG['synthetic_data_dir']}/train", f"{CONFIG['pseudo_save_dir']}/train"]
        val_dirs = [f"{CONFIG['synthetic_data_dir']}/val", f"{CONFIG['pseudo_save_dir']}/val"]
        
        train_ds = ASTMashupDatasetV2(train_dirs, CONFIG, is_train=True)
        val_ds = ASTMashupDatasetV2(val_dirs, CONFIG, is_train=False)
        
        train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=4)
        val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=4)
        
        print(f"Total Combined Training Samples: {len(train_ds)}")
        
        # Initialize V2 with V1's backbone
        v1_model = MessyMashupAST_v1.load_from_checkpoint(CONFIG['best_checkpoint_v1'])
        model = MessyMashupAST_v2(CONFIG, pretrained_v1_model=v1_model)
        
        checkpoint_cb = ModelCheckpoint(
            dirpath="checkpoints_v2", filename="best-ast-v2", monitor="val_macro_f1", mode="max", save_top_k=1
        )
        early_stop_cb = EarlyStopping(monitor="val_macro_f1", patience=CONFIG["patience"], mode="max", verbose=True)
        wandb_logger = WandbLogger(project=CONFIG["project_name"], name=CONFIG["experiment_name"])
        
        trainer = pl.Trainer(
            max_epochs=CONFIG["num_epochs"],
            accelerator="gpu",
            logger=wandb_logger,
            callbacks=[checkpoint_cb, early_stop_cb],
            accumulate_grad_batches=CONFIG["accumulate_grad"],
            precision="16-mixed",
            deterministic=True
        )
        
        trainer.fit(model, train_loader, val_loader)
        
    if RUN_INFERENCE:
        checkpoint_path = "checkpoints_v2/best-ast-v2.ckpt"
        if os.path.exists(checkpoint_path):
            print(f"Loading best V2 checkpoint: {checkpoint_path}")
            model = MessyMashupAST_v2.load_from_checkpoint(checkpoint_path)
        else:
            print("No V2 checkpoint found! Initializing untrained model.")
            model = MessyMashupAST_v2(CONFIG)
            
        run_inference_tta(model, CONFIG)